# Usage demonstration of MAST_tools utilities

Notebook demostrating the most relevant utilities available in the `MAST_tools` package.

Author: Rodrigo Ordonez-Hurtado (rodrigo.ordonez.hurtado@ibm.com)

In [ ]:
import sys
sys.path.insert(1, '../scripts')

import zarr
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

from pprint import pprint
from MAST_tools.store_utils import MASTStorageManager
from MAST_tools.signal_utils import MASTSignalManager
from MAST_tools.plotting_utils import MASTPlottingManager
from MAST_tools.MAST_dataset import MastDataset


# General settings

In [ ]:
# General settings for expepriments

LOCAL_MODE = True
TARGET_SHOT_ID = 30421  # Locally available: [30421, 30471]

# Other settings (not used in this demo):
# LEVEL = 2
# TEST_DATA = False
# VIA_PARQUET = False

# Settings for the StorageManager instance
STORE_MANAGER_SETTINGS = {
    "base_fsspec_protocol": "simplecache",
    "target_fsspec_protocol": "s3",
    "s3_endpoint_url": "https://s3.echo.stfc.ac.uk",
    "base_local_zarr_path": "/rds/project/rds-mOlK9qn0PlQ/fairmast/upload-tmp",
    "store_manager_id": "demo_storage_manager"
}

---
# Storage utilities

## Creation of store manager

In [ ]:
store_manager = MASTStorageManager(**STORE_MANAGER_SETTINGS)

print(f"Manager ID: {store_manager.store_manager_id}")

### List all available shot IDs

In [ ]:
if True:
    
    all_shots_ids = store_manager.list_all_shots(
        local=LOCAL_MODE,
    )
    
    pprint(all_shots_ids)


### List all sources

In [ ]:
if False:
    
    all_sources = store_manager.get_all_sources(
        shot_ids=[TARGET_SHOT_ID],  # all_shots_ids
        local=LOCAL_MODE,
    )

    try:
        pprint(all_sources)
        # print("\n\nall_sources.keys:\n")
        # pprint(all_sources.keys())
    except Exception as e:
        print(e)
        

### List all signals

In [ ]:
if False:
    
    all_signals = store_manager.get_all_signals(
        shot_ids=[TARGET_SHOT_ID],  # Use None for the entire dataset.
        local=LOCAL_MODE,
    )

    pprint(all_signals)


### Make shot store instance

**Store:** object for the physical implementation of the S3 storage service (a S3 bucket).

In [ ]:
store_ = None

if True:
    
    store_ = store_manager.make_shot_store(
        shot_info={
            "shot_id": TARGET_SHOT_ID,
            "local": LOCAL_MODE,
        }
    )
    
    print(f"Store: {store_}")
    
    # From the created store, a given group can be created
    # print(f'\n{xr.open_zarr(store=store_, group="pf_active")}')
    

### Make group from existing store instance

**Group:** the hierarchical container created to organise Zarr data.

In [ ]:
if False:
    
    # REMARK: It requires existing store "store_"
    
    # Create shot group from shot store
    group_from_store = store_manager.make_shot_group(
        data_origin=store_
    )
    print(f"group_from_store.tree():\n")
    print(group_from_store.tree())

if False:
    # Print group metadata
    
    gg = group_from_store.metadata.to_dict()
    pprint(gg["consolidated_metadata"]["metadata"])
        

### Make group directly from shot_id

In [ ]:
if False:
    
    group_from_shot_info = store_manager.make_shot_group(
        data_origin={
            "shot_id": TARGET_SHOT_ID,
            "local": LOCAL_MODE,
        }
    )
    print(f"group_from_shot_info.tree():\n")
    print(group_from_shot_info.tree())
    
if False:
    # Print group metadata
    
    gg = group_from_shot_info.metadata.to_dict()
    pprint(gg["consolidated_metadata"]["metadata"])

REMARK:
- Creation of group shot directly from shot info implicitly creates a store shot instance, which is easily accessible via the `.store` attribute.

In [ ]:
group_from_shot_info.store

### Check signal availability

REMARKS:
- A precomputed signal avaialbility file is required, which should be stored under the fairmast-data-preprocessing/metadata/ directory.
- One of such files is saved as "data_level2_signal_availability.csv" at fairmast-data-preprocessing/metadata/2025-04-17

In [ ]:
if False:
    
    dict_target_signals = {
        "thomson_scattering": ["n_e"],
        "spectrometer_visible": ["filter_spectrometer_bes_voltage"],
        "summary": ["power_nbi", "ip"],
    }

    print("\nSignals to be checked for simultaneous availability across all shots:")
    pprint(dict_target_signals)
    
    input("\nPress key to continue...")
    
    # signal_availability_file = "../../metadata/2025-04-17/data_level2_signal_availability.csv"
    signal_availability_file = "../metadata/2025-04-17/data_level2_signal_availability.csv"
    
    filtered_ids = store_manager.list_shots_by_signal_availability(
        availability_data_filepath=signal_availability_file,
        required_signals=dict_target_signals
    )

    print(f"\nfiltered_ids ({len(filtered_ids)} shots):")
    pprint(filtered_ids)

---
# Signal utilities

### Creation of signal manager

In [ ]:
signal_manager = MASTSignalManager(
    manager_id="demo_signal_manager",
    store_manager_settings=STORE_MANAGER_SETTINGS
)

print(f"Manager ID: {signal_manager.signal_manager_id}")

### Get signal values from existing store 

In [ ]:
if False:
    
    source_signal_name = "magnetics-flux_loop_flux"
    # source_signal_name = "thomson_scattering-n_e"  # -> Available signal, but full of nan values
    
    print(f"Target signal: {source_signal_name}\n")
    
    # First create a store from shot info
    store_from_shot_info = store_manager.make_shot_store(
        shot_info={
            "shot_id": TARGET_SHOT_ID,
            "local": LOCAL_MODE,
        }
    )
    
    # Then get signal values from that store
    source_name, signal_name = source_signal_name.split("-")
    signal_values = signal_manager.get_signal_values(
        data_origin=store_from_shot_info,  # Here, data_origin is an existing store
        source_name=source_name,
        signal_name=signal_name
    )

    print(f"Signal values:\n")
    pprint(signal_values)
    

### Get signal values from shot info

In [ ]:
if False:
    # Get signal values directly from shot info
    
    shot_info = {
        "shot_id": TARGET_SHOT_ID,
        "local": LOCAL_MODE,
    }
    
    source_signal_name = "magnetics-flux_loop_flux"
    # source_signal_name = "thomson_scattering-n_e"  # -> Available signal, but full of nan values
    
    source_name, signal_name = source_signal_name.split("-")
    signal_values = signal_manager.get_signal_values(
        data_origin=shot_info,  # Here, data_origin is a dictionary
        source_name=source_name,
        signal_name=signal_name
    )

    print(f"Signal values:\n")
    pprint(signal_values)
    

### Get time values and time type

REMARK: Not all the signals have the same time dimension, and so there is a function to facilitate its identification.

In [ ]:
if True:
    
    # source_signal_name = "magnetics-flux_loop_flux"  # -> Time
    source_signal_name = "magnetics-b_field_pol_probe_cc_field"  # -> Time mirnov
    
    source_name, signal_name = source_signal_name.split("-")
    time_vals, time_type = signal_manager.get_signal_times_and_time_type(
        signal_name=signal_name,
        data_origin=shot_info,
        source_name=source_name
    )
    
    print(f"Signal: {source_signal_name}")
    print(f"Time type: {time_type}")
    print(f"Time vals: {time_vals}")

### Get signal profile

**Signal profile:** Group instance for a given signal, with all the data & metadata.

In [ ]:
signal_profile = None

if False:
#     source_signal_name = "magnetics-flux_loop_flux"  # -> Time
    source_signal_name = "magnetics-b_field_pol_probe_cc_field"  # -> Time mirnov
    
    source_name, signal_name = source_signal_name.split("-")
    signal_profile = signal_manager.get_signal_profile(
        signal_name=signal_name,
        data_origin=shot_info,
        source_name=source_name
    )
    
#     print(f"\n{signal_profile}")
    
#     print(f"\n{signal_profile.time}")
#     print(f"\n{signal_profile.time_mirnov}")
#     print(f"\n{signal_profile.values}")
    

---
# Plotting utilities

## Creation of managers

In [ ]:
plotting_manager = MASTPlottingManager()

signal_manager = MASTSignalManager()


## Settings for tests

In [ ]:
%matplotlib notebook
# %matplotlib inline

# ----------------------------------------------------------------------------
# Create a shot group to plot from

group_from_shot_info = store_manager.make_shot_group(
    data_origin={
        "shot_id": TARGET_SHOT_ID,
        "local": LOCAL_MODE,
    }
)

available_sources = list(group_from_shot_info.keys())
target_source = 'summary'  # available_sources[0]

available_signals = list(group_from_shot_info[target_source].keys())


# ----------------------------------------------------------------------------
# Create source profiles from target store to plot from

source_profiles = signal_manager.get_source_profiles(
    data_origin=group_from_shot_info.store,
    source_name=target_source
)

# ----------------------------------------------------------------------------
if False:
    
    print(f"Available sources for shot {TARGET_SHOT_ID}:\n{available_sources}")
    
    print(f"\nAvailable signals for source '{target_source}':\n{available_signals}")

    print(f"\nSource profiles from source '{target_source}':")
    print(source_profiles)


## Plotting examples

### Plot single signal profile

In [ ]:
if False:
    
    # Select target signal from available signals
    target_signal = available_signals[0]
    print(f"Signal '{target_signal}' from source '{target_source}''.")
    
    # Plot signal
    plotting_manager.plot_1d_profiles(
        profiles=source_profiles[target_signal],
        fig_size=[8, 4]
    )


### Plot group of 1D profiles

In [ ]:
if False:
    print(f"1d profiles for source '{target_source}'.")
    plotting_manager.plot_1d_profiles(
        profiles=source_profiles,
        fig_size=[8, 8]
    )

### Plot target signal from store

In [ ]:
if False:
    
    source_signal_name = "magnetics-ip"
    
    source_name, signal_name = source_signal_name.split("-")
    plotting_manager.plot_signal(
        data_origin=group_from_shot_info.store,
        source_name=source_name,
        signal_name=signal_name,
        fig_size=[8, 4]
    )

### Plot target signal from shot info

In [ ]:
if False:
    
    source_signal_name = "thomson_scattering-n_e"
    # source_signal_name = "magnetics-ip"
    
    source_name, signal_name = source_signal_name.split("-")
    plotting_manager.plot_signal(
        data_origin={"shot_id": TARGET_SHOT_ID, "local": LOCAL_MODE},
        source_name=source_name,
        signal_name=signal_name,
        fig_size=[8, 4]
    )

### Plot entire signal group from store

In [ ]:
if False:
    
    source_name = "magnetics"
    # source_name = "thomson_scattering"
    
    plotting_manager.plot_group(
        data_origin=group_from_shot_info.store,
        source_name=source_name,
#         fig_size=[8, 12]
    )

### Plot signal group from shot info

In [ ]:
if False:
    
    # source_name = "magnetics"
    source_name = "thomson_scattering"
    
    plotting_manager.plot_group(
        data_origin={"shot_id": TARGET_SHOT_ID,"local": LOCAL_MODE},
        source_name=source_name,
#         fig_size=[8, 4]
    )

### Plot specific groups

In [ ]:
if False:
#     plotting_manager.plot_plasma_current(data_origin=group_from_shot_info.store)  # -> 1 signal
#     plotting_manager.plot_power_nbi(data_origin=group_from_shot_info.store)  # -> 1 signals
#     plotting_manager.plot_charge_exchange(data_origin=group_from_shot_info.store)  # -> 2 signals
#     plotting_manager.plot_thomson_scattering(data_origin=group_from_shot_info.store)  # -> 2 signals
#     plotting_manager.plot_spectrometer(data_origin=group_from_shot_info.store)  # -> 3 signals
#     plotting_manager.plot_magnetics(data_origin=group_from_shot_info.store)  # -> 11 signals

    pass

---
# MAST_Dataset utilities

## Settings for tets

In [ ]:
source_signal_list = [("thomson_scattering", "n_e"), ("magnetics", "flux_loop_flux")]
signal_transform_map = None
shot_transform = None
store_manager_settings = STORE_MANAGER_SETTINGS


## Create MastDataset instance

In [ ]:
test_dataset = MastDataset(
    local=LOCAL_MODE,
    shots_list=[30421, 30471],  # [30421, 30471], [TARGET_SHOT_ID]
    source_signal_list=source_signal_list,
    signal_level_transform_map=signal_transform_map,
    shot_level_transform=shot_transform,
    store_manager_settings=STORE_MANAGER_SETTINGS
)

test_dataset


## Check dataset attributes

In [ ]:
print(f"test_dataset.shots_list: {test_dataset.shots_list}")
print(f"test_dataset.source_signal_list: {test_dataset.source_signal_list}")

## Iterate over the created dataset

In [ ]:
if False:
    test_dataset.__getitem__(0)